# Bulan 1: Fondasi & Akuisisi Data
## Minggu 3-4 (Data Profiling)

Di tahap ini, kita bakal nganalisis seberapa "kotor" data mentah hasil *scraping* yang ada di `Gabungan.xlsx` sebelum kita bersihin di Bulan 2. Kita bakal ngecek data duplikat, nyari kata gaul yang sering muncul, dan ngitung *noise* kayak link atau mention.

In [ ]:
import pandas as pd
import re

# Load dataset mentah dari proses scraping
df = pd.read_excel('../Gabungan.xlsx')
print(f'Total baris data mentah: {len(df)}')
df[['full_text']].head()

### 1. Cek Persentase Data Duplikat
Kita cek seberapa banyak sih tweet atau data yang dobel (biasanya karena spam atau nge-retweet hal yang sama).

In [ ]:
total_data = len(df)
jumlah_duplikat = df.duplicated(subset=['full_text']).sum()
persentase_duplikat = (jumlah_duplikat / total_data) * 100

print(f'Jumlah data duplikat: {jumlah_duplikat} baris')
print(f'Persentase duplikat: {persentase_duplikat:.2f}%')

# Buang duplikat sementara buat analisis kata biar lebih akurat
df_unik = df.drop_duplicates(subset=['full_text'])

### 2. Mencari Kata Gaul (Slang Words) Paling Sering Muncul
Kita coba hitung seberapa sering kata-kata gaul/singkatan (kayak bgt, yg, aja, ga, laper, healing, dll) muncul di dataset kita.

In [ ]:
# Daftar kata gaul/slang yang mau kita intip frekuensinya
target_slang = ['bgt', 'yg', 'kalo', 'aja', 'ga', 'gak', 'udh', 'jd', 'tp', 'laper', 'healing', 'mo', 'aq']

# Gabungin semua teks jadi satu text panjang, terus di-lowercase
all_text = ' '.join(df_unik['full_text'].astype(str)).lower()

# Hitung kemunculannya
slang_counts = {}
for slang in target_slang:
    # Pake regex \b biar nyari kata yang pas (bukan bagian dari kata lain)
    count = len(re.findall(rf'\b{slang}\b', all_text))
    slang_counts[slang] = count

# Bikin dataframe biar tampilannya enak diliat (diurutin dari yang paling sering)
df_slang = pd.DataFrame(list(slang_counts.items()), columns=['Kata Gaul', 'Frekuensi'])
df_slang = df_slang.sort_values(by='Frekuensi', ascending=False).reset_index(drop=True)

print('Frekuensi Kemunculan Kata Gaul:')
display(df_slang)

### 3. Mengidentifikasi Noise (Link, Mention, Emoji/Simbol)
Seberapa banyak tweet yang mengandung link (URL), mention ke akun lain (@), atau simbol-simbol berlebihan yang bikin kotor?

In [ ]:
# Fungsi buat ngecek keberadaan noise di satu baris teks
def check_noise(text):
    text = str(text)
    has_link = bool(re.search(r'http\S+|www\.\S+', text))
    has_mention = bool(re.search(r'@\w+', text))
    # Anggap 'emoji berlebihan/simbol kotor' kalo teksnya banyak ngandung karakter non-alfanumerik
    # Kita cek sederhana aja, ada karakter spesial ga selain spasi/tanda baca biasa
    has_symbols = bool(re.search(r'[^\w\s.,!?\-]', text))
    return pd.Series([has_link, has_mention, has_symbols])

# Terapin ke dataframe kita
df_unik[['Punya_Link', 'Punya_Mention', 'Punya_Simbol']] = df_unik['full_text'].apply(check_noise)

# Hitung total dan persentasenya
total_unik = len(df_unik)
total_link = df_unik['Punya_Link'].sum()
total_mention = df_unik['Punya_Mention'].sum()
total_simbol = df_unik['Punya_Simbol'].sum()

print('--- Laporan Analisis Noise ---')
print(f'Terdapat Link / URL     : {total_link} tweet ({(total_link/total_unik)*100:.2f}%)')
print(f'Terdapat Mention Akun   : {total_mention} tweet ({(total_mention/total_unik)*100:.2f}%)')
print(f'Mengandung Simbol/Emoji : {total_simbol} tweet ({(total_simbol/total_unik)*100:.2f}%)')


### Kesimpulan Data Profiling
Dari hasil di atas, kelihatan jelas kalau data mentah kita lumayan "kotor". Banyak tweet yang mengandung noise (Link, Mention, Simbol) dan pemakaian kata gaul/singkatan yang sangat tinggi frekuensinya. 

Makanya, ini jadi alasan kuat kenapa kita **WAJIB** ngejalanin proses **Basic Cleaning & Advanced Normalization** di Bulan 2 nanti!